[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/InteligenciaArtificial/blob/main/tutoriales/notebooks/pydanticai/01-pydanticai-intro.ipynb)

# PydanticAI — Introducción práctica

Agentes tipados con PydanticAI: structured output, tools y streaming con Claude.

In [ ]:
# pip install pydantic-ai 'pydantic-ai[anthropic]'
import os
from pydantic_ai import Agent
from pydantic import BaseModel

MODELO = 'anthropic:claude-haiku-4-5-20251001'
print('PydanticAI listo')

## 1. Agente básico

In [ ]:
agente = Agent(
    model=MODELO,
    system_prompt='Eres un asistente de análisis legal. Responde en español.',
)

resultado = agente.run_sync('¿Qué es una cláusula de limitación de responsabilidad?')
print(resultado.data)
print(f'\nTokens usados: {resultado.usage().total_tokens}')

## 2. Structured output con Pydantic

In [ ]:
from typing import Literal

class AnalisisClausula(BaseModel):
    riesgo: Literal['alto', 'medio', 'bajo']
    tipo_clausula: str
    descripcion_riesgo: str
    recomendacion: str
    requiere_negociacion: bool

agente_analista = Agent(
    model=MODELO,
    result_type=AnalisisClausula,
    system_prompt='Analiza cláusulas contractuales. Sé específico y práctico.',
)

CLAUSULA = """
El proveedor podrá modificar unilateralmente las condiciones del servicio con un preaviso
de 24 horas. La responsabilidad máxima del proveedor queda limitada a 100€ en cualquier
caso, independientemente de los daños reales sufridos por el cliente.
"""

resultado = agente_analista.run_sync(CLAUSULA)
analisis = resultado.data

print(f'Tipo: {analisis.tipo_clausula}')
print(f'Riesgo: {analisis.riesgo.upper()}')
print(f'Riesgo descrito: {analisis.descripcion_riesgo}')
print(f'Recomendación: {analisis.recomendacion}')
print(f'Requiere negociación: {"Sí" if analisis.requiere_negociacion else "No"}')

## 3. Herramientas

In [ ]:
from datetime import date, timedelta

agente_plazos = Agent(
    model=MODELO,
    system_prompt='Ayudas a gestionar plazos y vencimientos de contratos.',
)

@agente_plazos.tool_plain
def calcular_fecha_preaviso(fecha_vencimiento: str, dias_preaviso: int) -> str:
    """Calcula la fecha límite para enviar un preaviso de no renovación."""
    vencimiento = date.fromisoformat(fecha_vencimiento)
    limite = vencimiento - timedelta(days=dias_preaviso)
    hoy = date.today()
    dias_restantes = (limite - hoy).days
    estado = 'URGENTE' if dias_restantes < 30 else ('próximo' if dias_restantes < 90 else 'con tiempo')
    return (
        f'Fecha límite de preaviso: {limite.isoformat()}. '
        f'Quedan {max(0, dias_restantes)} días ({estado}).'
    )

@agente_plazos.tool_plain
def listar_contratos_proximos(dias: int = 90) -> list[dict]:
    """Devuelve contratos que vencen en los próximos N días."""
    hoy = date.today()
    # Datos de ejemplo
    contratos = [
        {'id': 'C-001', 'proveedor': 'Acme Corp', 'vencimiento': (hoy + timedelta(days=45)).isoformat(), 'importe': 50000},
        {'id': 'C-002', 'proveedor': 'Tech SL',   'vencimiento': (hoy + timedelta(days=120)).isoformat(), 'importe': 12000},
        {'id': 'C-003', 'proveedor': 'Supplies SA','vencimiento': (hoy + timedelta(days=20)).isoformat(), 'importe': 8500},
    ]
    return [
        c for c in contratos
        if (date.fromisoformat(c['vencimiento']) - hoy).days <= dias
    ]

resultado = agente_plazos.run_sync(
    'Muéstrame los contratos que vencen pronto y calcula los plazos de preaviso (90 días) para cada uno.'
)
print(resultado.data)

## 4. Dependencias inyectadas

In [ ]:
from dataclasses import dataclass
from pydantic_ai import Agent, RunContext

@dataclass
class ConfiguracionEmpresa:
    nombre_empresa: str
    moneda: str
    jurisdiccion: str
    nivel_riesgo_maximo: str  # 'bajo', 'medio', 'alto'

agente_contextual = Agent(
    model=MODELO,
    deps_type=ConfiguracionEmpresa,
    system_prompt='Asesoras sobre contratos adaptando el análisis al contexto de la empresa.',
)

@agente_contextual.system_prompt
def prompt_dinamico(ctx: RunContext[ConfiguracionEmpresa]) -> str:
    """System prompt que se adapta a las dependencias."""
    cfg = ctx.deps
    return (
        f'Empresa: {cfg.nombre_empresa}. '
        f'Jurisdicción: {cfg.jurisdiccion}. '
        f'Moneda: {cfg.moneda}. '
        f'Nivel de riesgo máximo aceptable: {cfg.nivel_riesgo_maximo}. '
        f'Si el riesgo supera el nivel máximo, recomienda rechazar la cláusula.'
    )

# Configuración para una startup
cfg_startup = ConfiguracionEmpresa(
    nombre_empresa='TechStart SL',
    moneda='EUR',
    jurisdiccion='España',
    nivel_riesgo_maximo='bajo',
)

resultado = agente_contextual.run_sync(
    'El contrato de SaaS tiene una cláusula que permite al proveedor aumentar precios hasta un 30% anual.',
    deps=cfg_startup,
)
print(resultado.data)

## 5. Historial de conversación

In [ ]:
agente_chat = Agent(
    model=MODELO,
    system_prompt='Eres un asesor legal que recuerda el contexto de la conversación.',
)

# Turno 1
r1 = agente_chat.run_sync('El contrato con Acme Corp es por 500.000€ anuales y vence en agosto 2026.')
print('Turno 1:', r1.data)

# Turno 2 — pasa el historial del turno anterior
r2 = agente_chat.run_sync(
    '¿Cuánto perdemos si no renovamos ese contrato?',
    message_history=r1.new_messages(),
)
print('\nTurno 2:', r2.data)

# Turno 3
r3 = agente_chat.run_sync(
    'Dado ese importe, ¿qué nivel de prioridad tiene la renovación?',
    message_history=r2.all_messages(),
)
print('\nTurno 3:', r3.data)

print(f'\nHistorial total: {len(r3.all_messages())} mensajes')

## 6. Streaming

In [ ]:
import asyncio

agente_stream = Agent(
    model=MODELO,
    system_prompt='Redactas documentos legales claros y estructurados.',
)

async def generar_con_streaming(solicitud: str) -> None:
    print(f'Generando: {solicitud}\n')
    texto_completo = ''
    async with agente_stream.run_stream(solicitud) as stream:
        async for delta in stream.stream_text(delta=True):
            print(delta, end='', flush=True)
            texto_completo += delta
    print(f'\n\nTokens: {stream.usage().total_tokens}')
    return texto_completo

await generar_con_streaming(
    'Redacta un párrafo de cláusula de confidencialidad para un contrato de servicios SaaS.'
)